# LSTM Model Development
## Financial Fraud Detection System

This notebook focuses on building and training the LSTM model using the sequence datasets generated during sequence engineering.

### Key Design Decision
The model uses **time-based transaction sequences** created from globally ordered transaction data. Each sequence represents a pattern of consecutive financial transactions, allowing the LSTM model to learn temporal fraud behavior.

### Workflow
1. Load sequence datasets (`X_train`, `y_train`, etc.)
2. Build LSTM neural network architecture
3. Add dropout layers to reduce overfitting
4. Compile the model with optimizer and loss function
5. Train the model using training and validation datasets
6. Evaluate model performance on test data
7. Visualize training results and confusion matrix
8. Save trained LSTM model for future use

In [1]:
print("=" * 60)
print("LSTM MODEL DEVELOPMENT")
print("=" * 60)

# Data handling
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# Visualization
import matplotlib.pyplot as plt

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

# Evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow version:", tf.__version__)

LSTM MODEL DEVELOPMENT



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\AI_Project\AI_Financial_Fraud_Detection\env\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\AI_Project\AI_Financial_Fraud_Detection\env\Lib\site-packages\traitlets\config\application.py", line 1082, in launch_instance
    app.start()
  File "d:\AI_Project\AI_Financial_Fraud_Detection\env\Lib\site-packages\ipykernel\kernelapp.py", line 758, in start
    s

AttributeError: _ARRAY_API not found


[TensorFlow DLL Diagnostic] Analyzing: d:\AI_Project\AI_Financial_Fraud_Detection\env\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "d:\AI_Project\AI_Financial_Fraud_Detection\env\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
print("=" * 60)
print("BUILDING LSTM MODEL")
print("=" * 60)

timesteps = X_train.shape[1]
n_features = X_train.shape[2]

model = Sequential([

    # First LSTM layer
    LSTM(
        64,
        return_sequences=True,
        input_shape=(timesteps, n_features)
    ),

    BatchNormalization(),
    Dropout(0.3),

    # Second LSTM layer
    LSTM(32),

    BatchNormalization(),
    Dropout(0.3),

    # Dense layers
    Dense(16, activation='relu'),
    Dropout(0.2),

    # Output layer
    Dense(1, activation='sigmoid')
])

model.summary()

In [ ]:
print("=" * 60)
print("COMPILING MODEL")
print("=" * 60)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss='binary_crossentropy',

    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

print("Model compiled successfully!")

In [ ]:
print("=" * 60)
print("SETTING CALLBACKS")
print("=" * 60)

callbacks = [

    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    ),

    ModelCheckpoint(
        filepath=PROJECT_ROOT / "artifacts" / "models" / "best_lstm_model.keras",
        save_best_only=True
    )
]

print("Callbacks ready!")

In [ ]:
print("=" * 60)
print("TRAINING MODEL")
print("=" * 60)

history = model.fit(

    X_train,
    y_train,

    validation_data=(X_val, y_val),

    epochs=20,
    batch_size=64,

    callbacks=callbacks,

    verbose=1
)

In [ ]:
print("=" * 60)
print("MODEL EVALUATION")
print("=" * 60)

# Predictions
y_pred_prob = model.predict(X_test)

# Convert probabilities to 0/1
y_pred = (y_pred_prob > 0.5).astype(int)

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
print("=" * 60)
print("CONFUSION MATRIX")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred)

print(cm)

In [ ]:
print("=" * 60)
print("TRAINING VISUALIZATION")
print("=" * 60)

# Loss graph
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend(['Train', 'Validation'])

plt.show()

In [ ]:
print("=" * 60)
print("SAVING MODEL")
print("=" * 60)

MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model.save(MODEL_DIR / "final_lstm_model.keras")

print("Model saved successfully!")

In [ ]:
history_df = pd.DataFrame(history.history)

history_df.to_csv(
    MODEL_DIR / "training_history.csv",
    index=False
)

print("Training history saved!")